# ChatGeo · ตรวจน้ำท่วมจากดาวเทียม Sentinel-1 (ต้นแบบ v3)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Choonahakarn/chatgeo/blob/main/notebooks/flood_s1.ipynb)

**ทำอะไร:** หาภาพเรดาร์ Sentinel-1 ล่าสุดของพื้นที่ที่เลือก แล้วแยกน้ำเป็น 2 แบบ
- **น้ำผิดปกติ (แดง)** ตอนนี้มีน้ำ แต่ช่วงเดียวกันของปีก่อนๆ ส่วนใหญ่ไม่มี น่าจะเป็นน้ำท่วม
- **น้ำที่มีเกือบทุกปีช่วงนี้ (ฟ้าอ่อน)** เช่น นาข้าวที่ขังน้ำ ทุ่งรับน้ำ อาจท่วมจริงก็ได้ แต่ท่วมเป็นประจำ

ได้ออกมาเป็นไฟล์ GeoJSON สำหรับแผนที่ ChatGeo ภาพตัวอย่าง แผนที่แบบซูมได้ และสรุปรายจังหวัด

**ทำไมใช้เรดาร์:** ดาวเทียมเรดาร์เหมือนส่องไฟฉายลงมาเองแล้วรอรับแสงสะท้อน จึงถ่ายได้ทั้งกลางคืนและทะลุเมฆช่วงหน้าฝน ผิวน้ำเรียบเหมือนกระจก สะท้อนคลื่นออกไปทางอื่น ภาพตรงที่เป็นน้ำจึงมืด

**วิธีใช้:** เลือกพื้นที่ในเซลล์ตั้งค่า แล้วกดเมนู รันไทม์ › เรียกใช้ทั้งหมด (Runtime › Run all) รอราว 10–15 นาที ไม่ต้องมีบัญชีหรือ API key (ข้อมูลเปิดจาก Microsoft Planetary Computer)

In [ ]:
# ติดตั้งไลบรารี (ครั้งแรกราว 1 นาที)
!pip -q install pystac-client planetary-computer odc-stac rasterio scikit-image folium shapely pyproj

## ตั้งค่า

- `AREA` ชื่อย่อพื้นที่จาก `PRESETS` ด้านล่าง ถ้าอยากกำหนดเองให้ใส่ `'custom'` แล้วแก้ `AOI` กับ `AREA_NAME`
- `BASELINE_YEARS` เทียบกับช่วงเดียวกันของกี่ปีก่อน น้ำที่มีเกินครึ่งของปีเหล่านั้นนับเป็น "น้ำที่มีเกือบทุกปี" ใส่ `0` ถ้าไม่ต้องการเทียบ
- `RES` ความละเอียดเป็นเมตร 30 เร็วและพอดีกับพื้นที่ขนาดนี้
- `THRESHOLD` เส้นแบ่งน้ำ (เดซิเบล) ปล่อย `None` ให้คำนวณเอง
- `MAX_DAYS` ถ้าภาพวันล่าสุดถ่ายไม่ครบทั้งพื้นที่ จะเติมด้วยภาพวันก่อนหน้าได้สูงสุดกี่วัน

In [ ]:
PRESETS = {
    'ccs': ((100.8, 13.4, 101.6, 14.0), 'ฉะเชิงเทรา–ปราจีนบุรี (ลุ่มน้ำบางปะกง)'),
    'chi': ((102.9, 15.5, 104.6, 16.5), 'ลุ่มน้ำชีตอนล่าง (ร้อยเอ็ด–ยโสธร)'),
    'ayt': ((100.25, 14.1, 100.85, 14.75), 'อยุธยา–อ่างทอง (ลุ่มเจ้าพระยา)'),
    'skt': ((99.6, 16.8, 100.2, 17.4), 'สุโขทัย (ลุ่มน้ำยม)'),
    'ubn': ((104.6, 15.0, 105.3, 15.5), 'อุบลราชธานี (ลุ่มน้ำมูล)'),
}
AREA = 'ccs'                       # เปลี่ยนเป็นชื่อย่ออื่นใน PRESETS หรือ 'custom'
AOI = (100.8, 13.4, 101.6, 14.0)   # ใช้เมื่อ AREA = 'custom' (ลองจิจูดซ้าย, ละติจูดล่าง, ลองจิจูดขวา, ละติจูดบน)
AREA_NAME = 'พื้นที่ของฉัน'         # ใช้เมื่อ AREA = 'custom'
BASELINE_YEARS = 3                 # เทียบกับช่วงเดียวกันของ 3 ปีก่อน
DAYS_BACK = 24                     # หาภาพย้อนหลังกี่วัน
RES = 30                           # เมตรต่อพิกเซล
THRESHOLD = None                   # None = คำนวณเอง (Otsu)
MAX_DAYS = 4                       # เติมพื้นที่ที่ภาพล่าสุดไม่ครอบคลุมด้วยภาพวันก่อนหน้า

if AREA != 'custom':
    AOI, AREA_NAME = PRESETS[AREA]
CRS = None                         # None = เลือกโซน UTM ให้เอง

## ฟังก์ชันหลัก (รันเฉยๆ ไม่ต้องแก้)

In [ ]:
# ฟังก์ชันหลัก (ไม่ต้องแก้) · ต้นฉบับอยู่ที่ scripts/flood_core.py
import json
import math
import re
import urllib.request
from datetime import datetime, timedelta, timezone

import numpy as np
from scipy import ndimage


def scenes_by_day(client, aoi, start, end):
    """ภาพ Sentinel-1 ในช่วงวันที่ที่กำหนด จัดกลุ่มตามวัน เรียงจากใหม่ไปเก่า"""
    search = client.search(collections=['sentinel-1-rtc'], bbox=aoi,
                           datetime=start.strftime('%Y-%m-%d') + '/' + end.strftime('%Y-%m-%d'))
    by_day = {}
    for it in search.items():
        by_day.setdefault(it.datetime.strftime('%Y-%m-%d'), []).append(it)
    return sorted(by_day.items(), key=lambda x: x[0], reverse=True)


def utm_crs(aoi):
    """เลือกพิกัด UTM ที่ตรงกับกลางพื้นที่ (ไทยอยู่โซน 47 กับ 48)"""
    lon, lat = (aoi[0] + aoi[2]) / 2, (aoi[1] + aoi[3]) / 2
    return 'EPSG:%d' % ((32600 if lat >= 0 else 32700) + int((lon + 180) // 6) + 1)


def mosaic_latest(days, load_db, max_days=4, target=0.97):
    """ปะภาพหลายวันเข้าด้วยกัน: ใช้ภาพใหม่สุดก่อน ตรงไหนยังว่างค่อยเติมด้วยภาพวันก่อนหน้า
    load_db(items, like) ต้องคืน (db, ds) ที่อยู่บนกริดเดียวกัน
    คืนค่า (db, ds, วันที่ที่ใช้, สัดส่วนพื้นที่ที่มีข้อมูล, src) src บอกว่าแต่ละพิกเซลมาจากวันที่ลำดับไหนใน used (-1 = ไม่มีภาพ)"""
    db, ds, used, src = None, None, [], None
    for day, its in days[:max_days]:
        d, dsx = load_db(its, ds)
        if db is None:
            db, ds = d, dsx
            src = np.where(np.isfinite(db), 0, -1).astype('int8')
        else:
            hole = ~np.isfinite(db) & np.isfinite(d)
            if not hole.any():
                continue
            db[hole] = d[hole]
            src[hole] = len(used)
        used.append(day)
        if np.isfinite(db).mean() >= target:
            break
    if db is None:
        raise RuntimeError('ไม่พบภาพ Sentinel-1 ในช่วงนี้ ลองเพิ่ม DAYS_BACK')
    return db, ds, used, float(np.isfinite(db).mean()), src


def to_db(linear):
    """แปลงค่าการสะท้อนเรดาร์ (linear) เป็นเดซิเบล ค่าที่ใช้ไม่ได้เป็น NaN"""
    lin = np.asarray(linear, dtype='float32')
    out = np.full(lin.shape, np.nan, dtype='float32')
    ok = np.isfinite(lin) & (lin > 0)
    out[ok] = 10.0 * np.log10(lin[ok])
    return out


def speckle_filter(db, size=3):
    """ลดจุดรบกวนแบบเม็ดทราย (speckle) ด้วย median filter โดยไม่ทำให้ NaN ลาม"""
    filled = np.where(np.isfinite(db), db, 0.0)
    med = ndimage.median_filter(filled, size=size)
    return np.where(np.isfinite(db), med, np.nan).astype('float32')


def otsu_threshold(db, lo=-30.0, hi=0.0, clamp=(-24.0, -15.0), fallback=-19.0):
    """หาเส้นแบ่ง 'น้ำ' กับ 'ไม่ใช่น้ำ' จากฮิสโตแกรม (Otsu) แล้วจำกัดให้อยู่ในช่วงที่สมเหตุสมผล"""
    v = db[np.isfinite(db)]
    v = v[(v > lo) & (v < hi)]
    if v.size < 1000:
        return fallback
    try:
        from skimage.filters import threshold_otsu
        t = float(threshold_otsu(v))
    except Exception:  # noqa: BLE001
        return fallback
    return float(min(max(t, clamp[0]), clamp[1]))


def slope_degrees(dem, res):
    """ความชันของพื้นที่ (องศา) จาก DEM ที่อยู่ในพิกัดเมตร"""
    z = np.where(np.isfinite(dem), dem, np.nanmean(dem) if np.isfinite(dem).any() else 0)
    dy, dx = np.gradient(z, res)
    return np.degrees(np.arctan(np.hypot(dx, dy))).astype('float32')


def water_mask(db, threshold):
    """พิกเซลที่มืดกว่าเส้นแบ่ง = น้ำ (ตัดจุดเดี่ยวๆ ออก)"""
    w = np.isfinite(db) & (db < threshold)
    return ndimage.binary_opening(w, structure=np.ones((3, 3)))


def _drop_small(mask, min_pixels):
    """ตัดปื้นที่เล็กกว่า min_pixels พิกเซล"""
    lab, n = ndimage.label(mask)
    if not n:
        return mask
    sizes = ndimage.sum(mask, lab, index=np.arange(1, n + 1))
    keep = np.zeros(n + 1, dtype=bool)
    keep[1:] = sizes >= min_pixels
    return keep[lab]


def detect_flood(db, threshold, occurrence=None, slope=None, perm_occ=75, max_slope=5.0, min_pixels=10, reference_water=None):
    """น้ำท่วม = ตอนนี้เป็นน้ำ และปกติไม่ใช่แหล่งน้ำถาวร และไม่ใช่พื้นที่ลาดชัน (เงาภูเขาดูเหมือนน้ำ)
    ถ้าให้ reference_water (น้ำที่มีเป็นปกติช่วงนี้) จะตัดน้ำตามฤดูกาลออก เช่น นาข้าวที่ขังน้ำทุกปี"""
    water = water_mask(db, threshold)
    permanent = np.zeros_like(water) if occurrence is None else (np.nan_to_num(occurrence, nan=0) >= perm_occ)
    steep = np.zeros_like(water) if slope is None else (np.nan_to_num(slope, nan=0) > max_slope)
    flood = water & ~permanent & ~steep
    if reference_water is not None:
        # ขยายน้ำอ้างอิงออก 1 พิกเซล กันขอบขยับเล็กน้อยระหว่างสองภาพ
        flood &= ~ndimage.binary_dilation(reference_water, structure=np.ones((3, 3)))
    return _drop_small(flood, min_pixels), water, permanent, steep


def usual_water(masks, valids=None):
    """น้ำที่ "มีเป็นปกติ" ช่วงนี้ = เป็นน้ำเกินครึ่งของปีที่มีภาพ (เช่น 2 ใน 3 ปี)
    ปีที่ไม่มีภาพตรงพิกเซลนั้นไม่นับ จึงไม่ถูกซ่อนเพราะท่วมแค่ปีเดียว"""
    if not masks:
        return None
    cnt = np.zeros(masks[0].shape, 'int16')
    n = np.zeros(masks[0].shape, 'int16')
    for i, m in enumerate(masks):
        v = np.ones(m.shape, bool) if valids is None or valids[i] is None else valids[i]
        cnt += (m & v)
        n += v
    return (n > 0) & (cnt >= n // 2 + 1)


def classify(db, threshold, occurrence=None, slope=None, usual=None, perm_occ=75, max_slope=5.0, min_pixels=10):
    """แยกน้ำเป็น 2 ชั้น: flood = น้ำผิดปกติ (น่าจะท่วม), seasonal = น้ำที่มีเกือบทุกปีช่วงนี้ (นา ทุ่งรับน้ำ)"""
    flood, water, permanent, steep = detect_flood(db, threshold, occurrence, slope, perm_occ, max_slope, min_pixels, reference_water=usual)
    if usual is None:
        seasonal = np.zeros_like(flood)
    else:
        near_usual = ndimage.binary_dilation(usual, structure=np.ones((3, 3)))
        seasonal = _drop_small(water & ~permanent & ~steep & ~flood & near_usual, min_pixels)
    return {'water': water, 'permanent': permanent, 'steep': steep, 'flood': flood, 'seasonal': seasonal}


def round_coords(gj, nd=5):
    def r(c):
        return [r(x) for x in c] if isinstance(c[0], (list, tuple)) else [round(c[0], nd), round(c[1], nd)]
    for f in gj['features']:
        f['geometry']['coordinates'] = r(f['geometry']['coordinates'])
    return gj


def vectorize_layers(layers, transform, crs, res, src=None, dates=None, min_area_km2=0.05, max_features=4000,
                     big_km2=1.0, near_m=300, nd=4):
    """แปลงหลายชั้น (เช่น {'flood': ..., 'seasonal': ...}) เป็น GeoJSON ชุดเดียว
    แต่ละปื้นมี kind, area_km2, date (วันที่ของภาพที่ใช้มากที่สุดในปื้น)
    ชั้น flood มี conf: high = ผืนน้ำใหญ่ตั้งแต่ big_km2 หรืออยู่ห่างผืนใหญ่ไม่เกิน near_m เมตร
                       low  = ปื้นเล็กที่อยู่โดดๆ อาจเป็นนาที่เพิ่งปล่อยน้ำเข้าหรือบ่อ"""
    from rasterio.features import shapes
    from shapely.geometry import shape, mapping
    from shapely.ops import transform as shp_transform
    from shapely.strtree import STRtree
    from pyproj import Transformer
    tr = Transformer.from_crs(crs, 'EPSG:4326', always_xy=True).transform
    feats = []
    for kind, mask in layers.items():
        if mask is None or not mask.any():
            continue
        lab, n = ndimage.label(mask)
        day = None
        if src is not None and dates:
            k = len(dates)
            s = np.where(src[mask] >= 0, src[mask], k).astype('int64')
            counts = np.zeros((n + 1, k + 1), 'int64')
            np.add.at(counts, (lab[mask], s), 1)
            day = counts[:, :k].argmax(1)
        polys = []
        for geom, val in shapes(lab.astype('int32'), mask=mask, transform=transform):
            g = shape(geom)
            a = g.area / 1e6
            if a >= min_area_km2:
                polys.append((a, g, int(val)))
        polys.sort(key=lambda x: -x[0])
        polys = polys[:max_features]
        conf = {}
        if kind == 'flood':
            big = [g for a, g, _ in polys if a >= big_km2]
            tree = STRtree(big) if big else None
            for a, g, v in polys:
                hi = a >= big_km2 or (tree is not None and len(tree.query(g, predicate='dwithin', distance=near_m)) > 0)
                conf[v] = 'high' if hi else 'low'
        for a, g, v in polys:
            props = {'kind': kind, 'area_km2': round(a, 3)}
            if kind == 'flood':
                props['conf'] = conf[v]
            if day is not None:
                props['date'] = dates[int(day[v])]
            g4326 = shp_transform(tr, g.simplify(res, preserve_topology=True))
            feats.append({'type': 'Feature', 'properties': props,
                          'geometry': json.loads(json.dumps(mapping(g4326), default=float))})
    return round_coords({'type': 'FeatureCollection', 'features': feats}, nd)


def totals(gj):
    """พื้นที่รวม (ตร.กม.) แยกตามชนิดและความมั่นใจ"""
    t = {'flood_high': 0.0, 'flood_low': 0.0, 'seasonal': 0.0}
    for f in gj['features']:
        p = f['properties']
        key = 'seasonal' if p.get('kind') == 'seasonal' else ('flood_low' if p.get('conf') == 'low' else 'flood_high')
        t[key] += p['area_km2']
    return {k: round(v, 1) for k, v in t.items()}


def load_cg_th(url='https://raw.githubusercontent.com/Choonahakarn/chatgeo/main/data/th-regions.js', text=None):
    """อ่านขอบเขตภาค/จังหวัดจากไฟล์ th-regions.js ของ ChatGeo"""
    if text is None:
        with urllib.request.urlopen(url, timeout=60) as r:
            text = r.read().decode('utf-8')
    m = re.search(r'window\.CG_TH\s*=\s*(\{.*\})\s*;?\s*$', text, re.S)
    return json.loads(m.group(1)) if m else None


def province_stats(flood_gj, th):
    """พื้นที่ (ตร.กม.) แยกตามจังหวัด: [{'name', 'flood_high', 'flood_low', 'seasonal'}] เรียงตามน้ำท่วมที่มั่นใจ"""
    from shapely.geometry import shape
    from shapely.strtree import STRtree
    if not th or 'provinces' not in th:
        return []
    provs = [(f['properties']['name'], shape(f['geometry'])) for f in th['provinces']['features']]
    tree = STRtree([g for _, g in provs])
    tot = {}
    for f in flood_gj['features']:
        p = f['properties']
        key = 'seasonal' if p.get('kind') == 'seasonal' else ('flood_low' if p.get('conf') == 'low' else 'flood_high')
        c = shape(f['geometry']).representative_point()
        for i in tree.query(c):
            name, pg = provs[int(i)]
            if pg.contains(c):
                row = tot.setdefault(name, {'name': name, 'flood_high': 0.0, 'flood_low': 0.0, 'seasonal': 0.0})
                row[key] += p['area_km2']
                break
    rows = sorted(tot.values(), key=lambda r: (-r['flood_high'], -r['flood_low'], -r['seasonal']))
    for r in rows:
        for k in ('flood_high', 'flood_low', 'seasonal'):
            r[k] = round(r[k], 1)
    return rows

## 1) หาภาพเรดาร์ล่าสุด แล้วปะให้เต็มพื้นที่

ดาวเทียมถ่ายเป็นแถบ บางวันแถบเดียวไม่ครอบคลุมทั้งพื้นที่ จึงใช้ภาพวันล่าสุดก่อน ตรงไหนยังว่างค่อยเติมด้วยภาพวันก่อนหน้า และจำไว้ว่าแต่ละจุดมาจากภาพวันไหน

In [ ]:
import numpy as np, odc.stac, pystac_client, planetary_computer
from datetime import datetime, timedelta, timezone
CRS = CRS or utm_crs(AOI)
OUT = 'flood_s1_' + AREA
client = pystac_client.Client.open('https://planetarycomputer.microsoft.com/api/stac/v1',
                                   modifier=planetary_computer.sign_inplace)

def load_db(its, like=None):
    """โหลด VV ของภาพชุดหนึ่งบนกริดเดียวกัน แล้วแปลงเป็น dB และลดจุดรบกวน"""
    if like is None:
        ds = odc.stac.load(its, bands=['vv'], crs=CRS, resolution=RES, bbox=AOI, groupby='solar_day')
    else:
        ds = odc.stac.load(its, bands=['vv'], like=like, groupby='solar_day')
    vv = ds['vv'].max('time').values if ds.sizes.get('time', 1) > 1 else ds['vv'].isel(time=0).values
    return speckle_filter(to_db(vv)), (like if like is not None else ds)

end = datetime.now(timezone.utc)
start = end - timedelta(days=DAYS_BACK)
days = scenes_by_day(client, AOI, start, end)
print(AREA_NAME, '· พิกัด', CRS)
print('พบภาพ', len(days), 'วัน:', [d for d, _ in days])
db, ds, USED, cover, SRC = mosaic_latest(days, load_db, MAX_DAYS)
SCENES = {d: len(its) for d, its in days if d in USED}
DATE = USED[0]
DATE_RANGE = USED[-1] + ' ถึง ' + USED[0] if len(USED) > 1 else USED[0]
transform = ds.odc.geobox.affine
thr = THRESHOLD if THRESHOLD is not None else otsu_threshold(db)
print('ใช้ภาพวันที่', DATE_RANGE, '· มีข้อมูล %.0f%% ของพื้นที่' % (cover * 100), '· ขนาดภาพ', db.shape, '· เส้นแบ่งน้ำ %.1f dB' % thr)

## 2) โหลดข้อมูลประกอบ

- **แหล่งน้ำถาวร** จากสถิติ 30 ปีของ JRC ใช้ตัดแม่น้ำและหนองบึงที่มีน้ำเป็นปกติ
- **ความชัน** จาก Copernicus DEM ใช้ตัดเงาภูเขา ซึ่งในภาพเรดาร์ก็มืดเหมือนน้ำ
- **น้ำช่วงเดียวกันของปีก่อนๆ** (`BASELINE_YEARS` ปี) ใช้แยก "น้ำที่มีเกือบทุกปี" ออกจาก "น้ำผิดปกติ" ใช้หลายปีเพราะถ้าเทียบปีเดียว พื้นที่ที่บังเอิญท่วมปีที่แล้วด้วยจะถูกซ่อน

In [ ]:
def load_like(collection, band, valid):
    try:
        its = list(client.search(collections=[collection], bbox=AOI).items())
        if not its:
            return None
        da = odc.stac.load(its, bands=[band], like=ds)[band].astype('float32')
        return da.where(valid(da)).max('time').values
    except Exception as e:  # ถ้าโหลดไม่ได้ ข้ามขั้นนี้ไปก่อน
        print('โหลด', collection, 'ไม่ได้:', e)
        return None

occ = load_like('jrc-gsw', 'occurrence', lambda d: d <= 100)
dem = load_like('cop-dem-glo-30', 'data', lambda d: d > -1000)
slope = slope_degrees(dem, RES) if dem is not None else None

base_masks, base_valid, BASE_USED = [], [], []
for y in range(1, BASELINE_YEARS + 1):
    try:
        past = scenes_by_day(client, AOI, start - timedelta(days=365 * y), end - timedelta(days=365 * y))
        dbp, _, used_p, cov_p, _ = mosaic_latest(past, lambda its, like: load_db(its, ds), MAX_DAYS)
        base_masks.append(water_mask(dbp, thr)); base_valid.append(np.isfinite(dbp)); BASE_USED.append(used_p[0])
        print('ย้อนหลัง %d ปี: ภาพวันที่ %s · มีข้อมูล %.0f%%' % (y, used_p[0], cov_p * 100))
        del dbp
    except Exception as e:
        print('ย้อนหลัง %d ปี: โหลดไม่ได้ ข้ามปีนี้ (%s)' % (y, e))
usual = usual_water(base_masks, base_valid)
print('แหล่งน้ำถาวร:', 'มี' if occ is not None else 'ไม่มี', '· ความชัน:', 'มี' if slope is not None else 'ไม่มี',
      '· ปีที่ใช้เทียบ:', len(base_masks))

## 3) แยกน้ำผิดปกติกับน้ำที่มีเกือบทุกปี

- **น้ำผิดปกติ** = ตอนนี้เป็นน้ำ **และ** ไม่ใช่แหล่งน้ำถาวร **และ** ไม่ใช่ที่ลาดชัน **และ** ช่วงเดียวกันของปีก่อนๆ ส่วนใหญ่ไม่มีน้ำ
- **น้ำที่มีเกือบทุกปี** = ตอนนี้เป็นน้ำ และปีก่อนๆ ส่วนใหญ่ก็มีน้ำ (ไม่นับแหล่งน้ำถาวร)

In [ ]:
L = classify(db, thr, occ, slope, usual)
flood, seasonal, water, perm = L['flood'], L['seasonal'], L['water'], L['permanent']
px_km2 = RES * RES / 1e6
print('น้ำทั้งหมด %.1f ตร.กม. · น้ำถาวร %.1f · น้ำที่มีเกือบทุกปี %.1f · น้ำผิดปกติ %.1f' % (
    water.sum() * px_km2, (water & perm).sum() * px_km2, seasonal.sum() * px_km2, flood.sum() * px_km2))

In [ ]:
import matplotlib.pyplot as plt
fig, ax = plt.subplots(1, 2, figsize=(16, 7))
ax[0].imshow(db, cmap='gray', vmin=-25, vmax=0); ax[0].set_title('Sentinel-1 VV (dB) ' + ' / '.join(USED))
ax[1].imshow(db, cmap='gray', vmin=-25, vmax=0)
ov = np.zeros(db.shape + (4,)); ov[perm & water] = (0.2, 0.5, 1, 0.8)
ov[seasonal] = (0.45, 0.8, 1, 0.6)
ov[flood] = (1, 0.25, 0.2, 0.9)
ax[1].imshow(ov); ax[1].set_title('red = unusual water (likely flood), light blue = water most years, blue = permanent')
for a in ax: a.axis('off')
plt.tight_layout(); plt.savefig(OUT + '_' + DATE + '.png', dpi=110); plt.show()

## 4) แปลงเป็นรูปหลายเหลี่ยม (GeoJSON) และสรุปรายจังหวัด

ปื้นน้ำผิดปกติแบ่งความมั่นใจเป็น 2 ระดับ
- **สูง** ผืนน้ำตั้งแต่ 1 ตร.กม. หรือปื้นเล็กที่อยู่ห่างผืนใหญ่ไม่เกิน 300 ม. (ขอบของน้ำท่วมผืนเดียวกัน)
- **ต่ำ** ปื้นเล็กที่อยู่โดดๆ มักเป็นนาที่เพิ่งปล่อยน้ำเข้าหรือบ่อ

ขอบเขตจังหวัดดึงจากไฟล์ `data/th-regions.js` ของ ChatGeo บน GitHub

In [ ]:
gj = vectorize_layers({'flood': flood, 'seasonal': seasonal}, transform, CRS, RES, src=SRC, dates=USED)
T = totals(gj)
try:
    PROVS = province_stats(gj, load_cg_th())
except Exception as e:
    print('สรุปรายจังหวัดไม่ได้:', e); PROVS = []
gj['properties'] = {
    'v': 3, 'id': AREA, 'name': AREA_NAME, 'aoi': [round(x, 4) for x in AOI], 'date': DATE, 'dates': USED, 'scenes': SCENES,
    'baseline_years': len(base_masks), 'baseline_dates': BASE_USED, 'threshold_db': round(float(thr), 1),
    'totals': T, 'provinces': PROVS,
    'source': 'Sentinel-1 RTC (Copernicus/ESA) via Microsoft Planetary Computer; JRC Global Surface Water; Copernicus DEM',
    'method': 'ChatGeo v3: Otsu threshold on VV dB; minus JRC permanent water and slope > 5 deg; '
              'unusual = water not present in most of the previous %d years (same season)' % len(base_masks)}
with open(OUT + '_' + DATE + '.geojson', 'w') as f:
    json.dump(gj, f, ensure_ascii=False, separators=(',', ':'))
print('น้ำผิดปกติ มั่นใจสูง %.1f ตร.กม. · มั่นใจต่ำ %.1f · น้ำที่มีเกือบทุกปี %.1f · %d ปื้น' % (
    T['flood_high'], T['flood_low'], T['seasonal'], len(gj['features'])))
for r in PROVS[:10]:
    print('  จ.%s  ผิดปกติ %.1f (+ไม่แน่ใจ %.1f) · ทุกปี %.1f ตร.กม.' % (r['name'], r['flood_high'], r['flood_low'], r['seasonal']))

## 5) ดูบนแผนที่ เทียบกับสถานีวัดน้ำ ThaiWater

สามเหลี่ยมสีคือสถานีวัดน้ำจากข้อมูลที่งานเช้าของ ChatGeo เก็บไว้ ถ้าสถานีแถวไหนล้นตลิ่ง (แดง) แต่ดาวเทียมไม่เห็นน้ำ หรือกลับกัน คือจุดที่ควรตรวจดูให้ละเอียด

In [ ]:
import folium, urllib.request
cx, cy = (AOI[0] + AOI[2]) / 2, (AOI[1] + AOI[3]) / 2
# แผนที่พื้นหลังจาก Esri (เซิร์ฟเวอร์ OpenStreetMap ไม่ให้โหลดจาก Colab) สลับแผนที่/ดาวเทียมได้ที่มุมขวาบน
m = folium.Map(location=[cy, cx], zoom_start=9, tiles=None)
folium.TileLayer('https://server.arcgisonline.com/ArcGIS/rest/services/World_Street_Map/MapServer/tile/{z}/{y}/{x}',
                 attr='Esri, HERE, Garmin, © OpenStreetMap contributors', name='แผนที่ (Esri)').add_to(m)
folium.TileLayer('https://server.arcgisonline.com/ArcGIS/rest/services/World_Imagery/MapServer/tile/{z}/{y}/{x}',
                 attr='Esri, Maxar, Earthstar Geographics', name='ภาพดาวเทียม (Esri)').add_to(m)
def style(f):
    p = f['properties']
    if p['kind'] == 'seasonal':
        return {'color': '#62B6F0', 'weight': 0, 'fillColor': '#62B6F0', 'fillOpacity': 0.45}
    hi = p.get('conf') == 'high'
    return {'color': '#E5484D', 'weight': 1 if hi else 0, 'fillColor': '#E5484D', 'fillOpacity': 0.55 if hi else 0.25}
folium.GeoJson(gj, name='น้ำจาก Sentinel-1 ' + DATE_RANGE, style_function=style,
               tooltip=folium.GeoJsonTooltip(fields=['kind', 'conf', 'area_km2', 'date'],
                                             aliases=['ชนิด', 'ความมั่นใจ', 'พื้นที่ (ตร.กม.)', 'ภาพวันที่'])).add_to(m)
COL = {1: '#B5652B', 2: '#E0A526', 3: '#2FA36B', 4: '#2F7FE0', 5: '#E5484D'}
NAME = {1: 'น้อยวิกฤต', 2: 'น้อย', 3: 'ปกติ', 4: 'น้ำมาก', 5: 'ล้นตลิ่ง'}
try:
    live = json.load(urllib.request.urlopen('https://raw.githubusercontent.com/Choonahakarn/chatgeo/main/data/live.json', timeout=60))
    fg = folium.FeatureGroup(name='สถานีวัดน้ำ ThaiWater')
    for o in live.get('water', []):
        if AOI[0] <= o['lon'] <= AOI[2] and AOI[1] <= o['lat'] <= AOI[3]:
            folium.CircleMarker([o['lat'], o['lon']], radius=6, color='#fff', weight=1, fill=True, fill_opacity=0.95,
                                fill_color=COL.get(o['lv'], '#888'),
                                tooltip='%s (จ.%s) · %s · %s' % (o['name'], o['prov'], NAME.get(o['lv'], '?'), o['time'])).add_to(fg)
    fg.add_to(m)
except Exception as e:
    print('โหลดสถานี ThaiWater ไม่ได้:', e)
folium.Rectangle([[AOI[1], AOI[0]], [AOI[3], AOI[2]]], color='#555', weight=1, fill=False).add_to(m)
folium.LayerControl().add_to(m)
m

## ดาวน์โหลดผลลัพธ์

ได้ 2 ไฟล์: `flood_s1_<พื้นที่>_<วันที่>.geojson` (ไฟล์สำหรับแผนที่ ChatGeo) และ `.png` (ภาพตัวอย่าง)
ถ้าเบราว์เซอร์ถามว่าจะให้ดาวน์โหลดหลายไฟล์ไหม ให้กดอนุญาต หรือกดไอคอนโฟลเดอร์ทางซ้าย คลิกขวาที่ไฟล์ แล้วเลือกดาวน์โหลด

**เอาขึ้นเว็บ:** วางไฟล์ `.geojson` ไว้ในโฟลเดอร์ ChatGeo แล้วรัน `python3 scripts/add_flood.py <ไฟล์>` จากนั้นรัน `publish.command`

In [ ]:
try:
    from google.colab import files
    files.download(OUT + '_' + DATE + '.geojson')
    files.download(OUT + '_' + DATE + '.png')
except Exception:
    print('ไม่ได้รันใน Colab ไฟล์อยู่ในโฟลเดอร์เดียวกับโน้ตบุ๊กนี้')

## ข้อจำกัดของต้นแบบนี้ (สำคัญ)

- **เป็นต้นแบบเพื่อทดลอง ไม่ใช่ข้อมูลเตือนภัยทางการ** ให้เทียบกับ GISTDA และ ThaiWater เสมอ
- **สีฟ้าอ่อนไม่ได้แปลว่าไม่ท่วม** แปลว่าช่วงนี้ของปีก่อนๆ ก็มีน้ำ เช่น ทุ่งรับน้ำที่ท่วมเกือบทุกปี
- **ในเมืองมักมองไม่เห็นน้ำท่วม** เพราะคลื่นสะท้อนระหว่างตึกกับผิวน้ำแล้วกลับมาสว่าง
- **อาจนับผิดเป็นน้ำ** ในที่เรียบมาก เช่น ลานคอนกรีต ทราย และแปลงนาที่เพิ่งปล่อยน้ำเข้า (ส่วนใหญ่จะเป็นปื้น "มั่นใจต่ำ")
- **ภาพ Sentinel-1 มาทุกไม่กี่วัน** น้ำท่วมที่เกิดและลดระหว่างรอบถ่ายอาจไม่ถูกจับ

### ต่อยอดได้
- ฝึกโมเดล segmentation (เช่น U-Net) ด้วยชุดข้อมูลน้ำท่วมสาธารณะอย่าง Sen1Floods11 แทนการใช้เส้นแบ่งค่าเดียว
- ใช้รายงานจากผู้ใช้และสถานีวัดน้ำเป็นข้อมูลยืนยันผล แล้ววัดความแม่นยำ